In [47]:
# Cell 0 : setup.py (run first)

# --- Imports -------------------------------------------------------------
import copy
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Matplotlib inline utils for Jupyter
import matplotlib.pyplot as plt
from matplotlib import animation

from IPython.display import display

# --- Device switch -------------------------------------------------------
USE_GPU = False            # <<< flip to True when you move to the cluster
device  = torch.device('cuda' if USE_GPU and torch.cuda.is_available()
                       else 'cpu')
print(f"Running on: {device}")

# --- Game constants ------------------------------------------------------
τ   = 0.5                         # time step (s)
T   = 1.0                         # horizon (s)
K   = int(T / τ)                  # number of steps
I   = 2                           # number of potential targets

BOX_POS = 2.0                     # |position| ≤ 2 m
BOX_VEL = 2.0                     # |velocity| ≤ 1 m/s
BOX_ACC = 2.0                     # |acceleration| ≤ 2 m/s²   (action bounds)

Z_TARGETS = torch.tensor([[0.0,  1.0, 0.0, 0.0],
                          [0.0, -1.0, 0.0, 0.0]], device=device)   # (I,4)

Kmat = torch.diag(torch.tensor([1.0, 1.0, 0.0, 0.0], device=device))
R1   = torch.diag(torch.tensor([0.0, 0.0], device=device))
R2   = torch.diag(torch.tensor([1.0, 1.0], device=device))

# --- Training parameters -------------------------------------------------------------
BELIEF_DIM  = 1                 # because I = 2  →  simplex is 1-D
FEAT_DIM    = 1 + 8 + BELIEF_DIM
BETA_RNAD   = 1000.             # smaller → stronger reg in R-NAD reward
INNER_EPOCH = 20000             # epochs of PG between reference updates
OUTER_ITER  = 10               # how many ref updates you want

# --- Utility -------------------------------------------------------------
def clip(a: torch.Tensor, lo: float, hi: float) -> torch.Tensor:
    return torch.clamp(a, lo, hi)

Running on: cpu


In [48]:
# Cell 1 : hexners_env.py

class HexnersGameEnv:
    """
    Continuous-state, discrete-time wrapper for Hexner's 2p0s differential game.
    State  x = [p1_x, p1_y, p1_vx, p1_vy,
                p2_x, p2_y, p2_vx, p2_vy] ∈ ℝ⁸
    Public belief p ∈ Δ(I).  First player privately knows the true target i*.
    """
    def __init__(self, batch_size: int = 1, device: torch.device = device):
        self.batch_size = batch_size
        self.device     = device
        self.reset()
    
    # ------------------------------------------------------------
    def reset(self):
        """Sample a fresh game from p₀, zero state, return observations."""
        # Nature draws target indices for each parallel game in batch
        p0 = torch.full((self.batch_size, I), 1.0 / I,
                        device=self.device)           # uniform prior
        self.i_star = torch.multinomial(p0, 1).squeeze(-1)   # (B,)
        
        # Initial public belief = p₀
        self.p = p0.clone()
        
        # Initial positions and velocities = 0
        self.x = torch.zeros(self.batch_size, 8, device=self.device)
        self.t = torch.zeros(self.batch_size, device=self.device)   # current time
        self.step_idx = 0
        return self._get_obs()
    
    # ------------------------------------------------------------
    def _f(self, x: torch.Tensor,
        u1: torch.Tensor, u2: torch.Tensor) -> torch.Tensor:
        """
        Continuous-time dynamics  ẋ = f(t,x,u1,u2)  for a double integrator.
        Implemented *without* in-place writes to keep Autograd happy.
        """
        v1 = x[:, 2:4]            # p1 velocity
        v2 = x[:, 6:8]            # p2 velocity
        #     [p1_posdot, p1_acc , p2_posdot, p2_acc ]  ← concat along last dim
        return torch.cat([v1, u1, v2, u2], dim=-1)     # shape (B,8)
    
    # ------------------------------------------------------------
    @torch.no_grad()
    def step(self,
            u1: torch.Tensor, u2: torch.Tensor,
            action_idx_p1: torch.Tensor) -> Dict[str, Any]:
        """
        Advance one τ step with second-order integration:
            p ← p + v*τ + ½ a τ²
            v ← v + a τ
        No autograd tracking (REINFORCE does not need it here).
        """
        # -------- clip controls ------------------------------------
        u1 = torch.clamp(u1, -BOX_ACC, BOX_ACC)          # (B,2)
        u2 = torch.clamp(u2, -BOX_ACC, BOX_ACC)

        # -------- unpack current state -----------------------------
        pos1 = self.x[:, 0:2]        # (B,2)
        vel1 = self.x[:, 2:4]
        pos2 = self.x[:, 4:6]
        vel2 = self.x[:, 6:8]

        # -------- second-order update ------------------------------
        pos1_new = pos1 + vel1 * τ + 0.5 * u1 * (τ ** 2)
        vel1_new = vel1 + u1 * τ
        pos2_new = pos2 + vel2 * τ + 0.5 * u2 * (τ ** 2)
        vel2_new = vel2 + u2 * τ

        # -------- hard bounds --------------------------------------
        pos1_new = pos1_new.clamp(-BOX_POS, BOX_POS)
        vel1_new = vel1_new.clamp(-BOX_VEL, BOX_VEL)
        pos2_new = pos2_new.clamp(-BOX_POS, BOX_POS)
        vel2_new = vel2_new.clamp(-BOX_VEL, BOX_VEL)

        # -------- commit new state ---------------------------------
        self.x = torch.cat([pos1_new, vel1_new, pos2_new, vel2_new], dim=-1)

        # -------- running cost (unchanged) -------------------------
        u1_cost = (u1.unsqueeze(1) @ R1 @ u1.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        u2_cost = (u2.unsqueeze(1) @ R2 @ u2.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        l_running = 0.5 * (u1_cost - u2_cost) * τ        # (B,)

        # -------- bookkeeping --------------------------------------
        self.last_action_idx = action_idx_p1
        self.t += τ
        self.step_idx += 1
        done = (self.step_idx >= K)

        return {
            "state": self._get_obs(),
            "running_cost": l_running,
            "done": done
        }
    
    # ------------------------------------------------------------
    def _terminal_cost(self) -> torch.Tensor:
        """
        g_i(x_K) using private i* for each trajectory.
        Positive for P1 (minimiser), negative for P2 (maximiser).
        """
        x1 = self.x[:, 0:4]                       # p1 pos+vel
        x2 = self.x[:, 4:8]                       # p2 pos+vel
        
        deltas1 = x1 - Z_TARGETS[self.i_star]     # (B,4)
        deltas2 = x2 - Z_TARGETS[self.i_star]
        term = 0.5 * (
            (deltas1 @ Kmat * deltas1).sum(dim=-1) -
            (deltas2 @ Kmat * deltas2).sum(dim=-1)
        )
        return term                               # (B,)
    
    # ------------------------------------------------------------
    def _get_obs(self) -> Dict[str, torch.Tensor]:
        """Return public observation for *both* players."""
        # P1 additionally gets true target index (handled by trainer)
        return {"t": self.t.clone(),
                "x": self.x.clone(),
                "p": self.p.clone()}

In [49]:
# Cell 2 : networks.py
# ---------------------------------------------------------------
#  Policy parameterisations for Hexner's game (option B: finite
#  deterministic action prototypes + categorical mixture)
# ---------------------------------------------------------------

class P1Policy(nn.Module):
    """
    ϵ‐greedy-free stochastic policy for Player 1.

    • Inputs  : obs  = { "t":  (B,),
                         "x":  (B, 8),
                         "p":  (B, I) }      public belief
               : i_star (B,)  private target indices ∈ {0,…,I−1}

    • Internals:
          η_θ(t,x,p)  →  A_logits  (B, I, I)
                         u1_proto   (B, I, 2)
      – Row i of A is the categorical probabilities when the true
        target is i.  Each row is softmax-normalised on forward().
      – u1_proto[j] is the 2-D deterministic prototype for column j.
        Bound is enforced by tanh → scale to [−2,2].

    • forward() returns triple:
         u1_action  (B,2)   – sampled control used in env
         logπ       (B,)    – log-prob for policy gradient
         misc       dict    – { "A": (B,I,I), "j":(B,) }
    """
    def __init__(self, hidden: int = 128):
        super().__init__()
        inp_dim = FEAT_DIM             
        
        self.net = nn.Sequential(
            nn.Linear(inp_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        # Output heads
        self.head_A   = nn.Linear(hidden, I * I)     # logits, no softmax yet
        self.head_u1  = nn.Linear(hidden, I * 2)     # raw prototypes
        self.head_logstd = nn.Parameter(torch.full((1, 1, 2), -1.0))  # σ ≈ e⁻¹ ≈ 0.37

    # ------------------------------------------------------------
    def forward(self, obs: Dict[str, torch.Tensor],
                i_star: torch.Tensor,
                temperature: float = 1.0):
        B = obs["x"].shape[0]
        batch_idx = torch.arange(B, device=obs["x"].device)   # ← NEW

        h = torch.cat([obs["t"].unsqueeze(-1),      # (B,1)
                       obs["x"],                    # (B,8)
                       obs["p"][..., :1]], dim=-1)  # (B,I-1)
        h = self.net(h)                             # (B,H)
        
        # --- Mixture matrix A (B,I,I) ---------------------------
        A_logits = self.head_A(h).view(B, I, I) / temperature
        A        = F.softmax(A_logits, dim=-1)      # row softmax
        
        # --- Action prototypes u1 (B,I,2) -----------------------
        u1_raw   = self.head_u1(h).view(B, I, 2)
        u1_proto = torch.tanh(u1_raw) * BOX_ACC     # bound in [−2,2]
        
        LOG_STD_MIN, LOG_STD_MAX = -3.0, 1.0          # 0.05 ≤ σ ≤ 2.7
        
        log_std = self.head_logstd.expand(B, I, 2)        # tied variance
        log_std = log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        std     = torch.exp(log_std)
        inv_var = torch.exp(-2 * log_std)              # safe and finite

        # --- Sample action column j given private i* -----------
        row_probs = A.gather(1, i_star.view(B,1,1).expand(-1,1,I))\
                      .squeeze(1)                   # (B,I)

        if torch.isnan(row_probs).any():
            print("A_logits has NaN:", torch.isnan(A_logits).any().item())
            print("Temperature =", temperature)          # if you changed it
            print("obs['p'] contains NaN:", torch.isnan(obs['p']).any().item())
            raise ValueError("NaNs detected -- aborting early to inspect.")

        dist_cat = torch.distributions.Categorical(probs=row_probs)
        j        = dist_cat.sample()
        # Gaussian sample
        eps      = torch.randn_like(std[:,0,:])
        u1_action= u1_proto[batch_idx, j] + std[batch_idx, j] * eps

        # log π_live(a)
        log_prob_gauss = -0.5*((eps)**2 + math.log(2*math.pi)) - log_std[batch_idx, j]
        logp           = dist_cat.log_prob(j) + log_prob_gauss.sum(-1)
        
        misc = {"A": A,            # needed for Bayes update
                "j": j,
                "u1_proto": u1_proto}
        return u1_action, logp, misc


class P2Policy(nn.Module):
    """
    Stochastic policy for Player 2.

    Inputs (public only): same obs dict as P1 (no private info).
    Output:
        B_logits  (B,I+1) → softmax→B   categorical over I+1 actions
        u2_proto  (B,I+1,2) deterministic accelerations
    Returns (u2_action, logπ, misc)
    """
    def __init__(self, hidden: int = 128):
        super().__init__()
        inp_dim = FEAT_DIM                 # 10
        self.net = nn.Sequential(
            nn.Linear(inp_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
        )
        self.head_B      = nn.Linear(hidden, I + 1)      # mixture weights
        self.head_mu     = nn.Linear(hidden, (I + 1) * 2)
        self.head_logstd = nn.Parameter(torch.full((1, 1, 2), -1.0))

    # ------------------------------------------------------------
    def forward(self, obs: Dict[str, torch.Tensor], temperature: float = 1.0):
        B = obs["x"].shape[0]
        batch_idx = torch.arange(B, device=obs["x"].device)

        h = torch.cat([
                obs["t"].unsqueeze(-1),
                obs["x"],
                obs["p"][..., :1]
            ], dim=-1)
        h = self.net(h)

        B_logits = self.head_B(h) / temperature           # (B,I+1)
        mix_prob = F.softmax(B_logits, dim=-1)

        mu  = torch.tanh(self.head_mu(h).view(B, I + 1, 2)) * BOX_ACC

        LOG_STD_MIN, LOG_STD_MAX = -3.0, 1.0          # 0.05 ≤ σ ≤ 2.7
        
        log_std = self.head_logstd.expand(B, I + 1, 2)
        log_std = log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        std     = torch.exp(log_std)
        inv_var = torch.exp(-2 * log_std)              # safe and finite

        comp_dist = torch.distributions.Categorical(probs=mix_prob)
        k         = comp_dist.sample()                    # (B,)

        eps       = torch.randn_like(std[:, 0, :])
        u2_action = mu[batch_idx, k] + std[batch_idx, k] * eps

        logp_cat   = comp_dist.log_prob(k)
        logp_gauss = -0.5 * (eps**2 + math.log(2*math.pi)) \
                     - log_std[batch_idx, k]
        logp       = logp_cat + logp_gauss.sum(-1)

        misc = {"B": mix_prob, "k": k, "mu": mu, "std": std}
        return u2_action, logp, misc


class ValueNet(nn.Module):
    """Shared critic V(t,x,p) ≈ game value from public state."""
    def __init__(self, hidden: int = 128):
        super().__init__()
        inp_dim = FEAT_DIM
        self.net = nn.Sequential(
            nn.Linear(inp_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, obs: Dict[str, torch.Tensor]) -> torch.Tensor:
        h = torch.cat([obs["t"].unsqueeze(-1), obs["x"], obs["p"][..., :1]], dim=-1)
        return self.net(h).squeeze(-1)          # (B,)
    

# ---------------------------------------------------------------
def bayes_update(p: torch.Tensor,          # (B,I)
                 A: torch.Tensor,          # (B,I,I)
                 j: torch.Tensor) -> torch.Tensor:
    """
    Vectorised Bayes rule:
      p'[i] = A[i,j] * p[i] / Σ_{i'} A[i',j]p[i'] .
    """
    Bsz = p.shape[0]
    batch_idx = torch.arange(Bsz, device=device)
    Aj        = A[batch_idx, :, j]                    # (B,I) gather column j
    numer     = Aj * p                                # element-wise
    denom     = numer.sum(dim=-1, keepdim=True) + 1e-8
    return numer / denom                             # (B,I)


@torch.no_grad()
def logprob_p1_cont(policy_ref, obs, i_star, a):
    B = a.shape[0]
    h = torch.cat([obs["t"].unsqueeze(-1), obs["x"], obs["p"][..., :1]], dim=-1)
    h = policy_ref.net(h)
    A  = F.softmax(policy_ref.head_A(h).view(B, I, I), dim=-1)
    mu = torch.tanh(policy_ref.head_u1(h).view(B, I, 2)) * BOX_ACC
    log_std = policy_ref.head_logstd.expand(B, I, 2)
    inv_var = torch.exp(-2*log_std)             # 1/σ²

    diff = a.unsqueeze(1) - mu                  # (B,I,2)
    log_gauss = -0.5 * ((diff**2)*inv_var).sum(-1) \
                - log_std.sum(-1) - math.log(2*math.pi)
    # mixture
    log_mix = torch.logsumexp(torch.log(A.gather(1,
                             i_star.view(B,1,1).expand(-1,1,I)).squeeze(1))
                             + log_gauss, dim=-1)
    return log_mix     # (B,)


@torch.no_grad()
def logprob_p2_cont(policy_ref: P2Policy,
                    obs: Dict[str, torch.Tensor],
                    a: torch.Tensor) -> torch.Tensor:
    """
    log π_ref(a | obs)  for Player-2 Gaussian mixture.
    """
    B = a.shape[0]
    h = torch.cat([
            obs["t"].unsqueeze(-1),
            obs["x"],
            obs["p"][..., :1]
        ], dim=-1)
    h = policy_ref.net(h)

    mix_logits = policy_ref.head_B(h)
    mix_prob   = F.softmax(mix_logits, dim=-1)            # (B,I+1)

    mu  = torch.tanh(policy_ref.head_mu(h).view(B, I + 1, 2)) * BOX_ACC
    log_std = policy_ref.head_logstd.expand(B, I + 1, 2)
    inv_var = torch.exp(-2 * log_std)

    diff = a.unsqueeze(1) - mu                           # (B,I+1,2)
    log_gauss = -0.5 * ((diff**2) * inv_var).sum(-1) \
                - log_std.sum(-1) - math.log(2*math.pi)

    log_mix = torch.logsumexp(torch.log(mix_prob) + log_gauss, dim=-1)
    return log_mix            # (B,)

In [50]:
# Cell 3 : rnad_pg_trainer.py
# ---------------------------------------------------------------
class RNADTrainer:
    """
    Two-level R-NaD:
      inner  – actor-critic on ℓ̃  (policy-dependent running cost)
      outer  – every `INNER_EPOCH` steps copy live → reference
    """
    def __init__(self,
                 env_factory,
                 p1_live: P1Policy, p2_live: P2Policy,
                 value_net: ValueNet,
                 beta=BETA_RNAD,
                 lr=1e-4, 
                 entropy_beta=(0,0)        # no need for additional entropy regularization in default R-NaD
                 ):      
        # live / reference policies
        self.p1      = p1_live.to(device)
        self.p2      = p2_live.to(device)
        self.p1_ref  = copy.deepcopy(self.p1).eval().requires_grad_(False)
        self.p2_ref  = copy.deepcopy(self.p2).eval().requires_grad_(False)

        self.V       = value_net.to(device)
        self.env_factory = env_factory
        self.beta    = beta
        self.β1, self.β2 = entropy_beta

        # self.opt_pi  = torch.optim.Adam(
        #                    list(self.p1.parameters()) +
        #                    list(self.p2.parameters()), lr=lr)
        self.opt_v   = torch.optim.Adam(self.V.parameters(), lr=lr)
        
        logstd_params  = [p for n, p in self.p1.named_parameters()
                        if "head_logstd" in n] + \
                        [p for n, p in self.p2.named_parameters()
                        if "head_logstd" in n]

        other_params   = [p for n, p in self.p1.named_parameters()
                        if "head_logstd" not in n] + \
                        [p for n, p in self.p2.named_parameters()
                        if "head_logstd" not in n]
        # freeze log_std: set requires_grad=False *or* give them lr=0
        for p in logstd_params:
            p.requires_grad_(False)

        self.opt_pi = torch.optim.Adam(other_params, lr=lr)

        total_inner_steps = OUTER_ITER * INNER_EPOCH
        warmup_inner      = int(0.05 * total_inner_steps)   # 5 % warm-up
        
        self.pi_sched = CosineWithWarmup(self.opt_pi,
                                        warmup_steps=warmup_inner,
                                        total_steps=total_inner_steps,
                                        lr_min=1e-5)

        self.v_sched  = CosineWithWarmup(self.opt_v,
                                        warmup_steps=warmup_inner,
                                        total_steps=total_inner_steps,
                                        lr_min=1e-5)


    # ------------------------------------------------------------
    def _rollout_batch(self, B):
        """One rollout with regularised cost."""
        env   = self.env_factory()
        env.batch_size = B
        obs   = env.reset()
        i_st  = env.i_star

        logs_p1, logs_p2  = [], []
        logs_ref1, logs_ref2 = [], []
        ent1, ent2, costs = [], [], []
        values            = []

        for k in range(K):
            a1, lp1, m1 = self.p1(obs, i_st)
            a2, lp2, m2 = self.p2(obs)

            # reference log-probs (no grad)
            lp1_ref = logprob_p1_cont(self.p1_ref, obs, i_st, a1.detach())
            lp2_ref = logprob_p2_cont(self.p2_ref, obs,       a2.detach())

            step   = env.step(a1, a2, m1["j"])
            env.p  = bayes_update(env.p, m1["A"], m1["j"])
            obs    = step["state"]

            # Regularised running cost (scalar, +ve for P1)
            reg = (lp1.detach() - lp1_ref -
                   lp2.detach() + lp2_ref) / self.beta
            costs.append(step["running_cost"] + reg)

            # logging
            logs_p1.append(lp1);   logs_p2.append(lp2)
            logs_ref1.append(lp1_ref); logs_ref2.append(lp2_ref)

            row_probs = m1["A"].gather(               # (B,1,I)
                1, i_st.view(B, 1, 1).expand(-1, 1, I)
            ).squeeze(1)                              # (B, I)

            # Player-1 entropy
            e1 = -(row_probs * torch.log(row_probs + 1e-8)).sum(-1)   # (B,)

            # Player-2 entropy (unchanged)
            e2 = -(m2["B"] * torch.log(m2["B"] + 1e-8)).sum(-1)
            ent1.append(e1); ent2.append(e2)

            values.append(self.V(obs))

        G = torch.stack(costs, dim=0).sum(0) + env._terminal_cost()
        V0 = values[0].squeeze(-1)
        Adv = (G - V0).detach()

        std_adv = Adv.std()
        if std_adv > 1e-3:
            Adv = (Adv - Adv.mean()) / (std_adv + 1e-6)
        else:                       # nearly constant – only centre
            Adv = Adv - Adv.mean()
        Adv = Adv.clamp(-10., 10.)  # hard cap

        return dict(
            Adv=Adv, G=G.detach(), V0=V0,
            logp1=torch.stack(logs_p1, dim=0),
            logp2=torch.stack(logs_p2, dim=0),
            entropy1=torch.stack(ent1, dim=0),
            entropy2=torch.stack(ent2, dim=0)
        )

    # ------------------------------------------------------------
    def inner_step(self, batch_size=512, clip_grad=0.5):        # clip value reduced from 5.
        roll = self._rollout_batch(batch_size)

        # critic update
        v_loss = F.mse_loss(roll["V0"], roll["G"])
        self.opt_v.zero_grad(); v_loss.backward(); self.opt_v.step()

        # policy loss
        ent1 = roll["entropy1"].sum(0); ent2 = roll["entropy2"].sum(0)
        J1 = (roll["logp1"].sum(0) * roll["Adv"]).mean() - self.β1*ent1.mean()
        J2 = -(roll["logp2"].sum(0) * roll["Adv"]).mean() - self.β2*ent2.mean()
        loss_pi = J1 + J2

        self.opt_pi.zero_grad()
        loss_pi.backward()


        # debug NaN issue
        # --- NaN / Inf guard ------------------------------------------
        for n, p in self.p1.named_parameters():
            if p.grad is not None and torch.isnan(p.grad).any():
                print("NaN in grad of", n)
                raise RuntimeError("gradient NaN before clipping")

        total_norm = torch.nn.utils.clip_grad_norm_(
                        self.p1.parameters(), clip_grad, error_if_nonfinite=False)
        if not torch.isfinite(total_norm):
            print("grad-norm =", total_norm)
            raise RuntimeError("Inf grad-norm before optimiser.step")


        torch.nn.utils.clip_grad_norm_(self.p1.parameters(), clip_grad)
        torch.nn.utils.clip_grad_norm_(self.p2.parameters(), clip_grad)
        self.opt_pi.step()

        self.pi_sched.step()
        self.v_sched.step()

        return dict(policy_loss=loss_pi.item(), v_loss=v_loss.item(),
                    adv=roll["Adv"].mean().item())

    # ------------------------------------------------------------
    def train(self,
            outer_iter=OUTER_ITER,
            inner_epochs=INNER_EPOCH,
            batch_size=512,
            visualize_every=1):           # NEW
        for outer in range(outer_iter):
            for ep in range(inner_epochs):
                stats = self.inner_step(batch_size)
                if ep % 50 == 0:
                    print(f"[outer {outer} | ep {ep}] "
                        f"Lπ={stats['policy_loss']:+.3f}  "
                        f"LV={stats['v_loss']:.3f}  "
                        f"Adv={stats['adv']:+.4f}")

            # ----- outer update: freeze new reference -----
            self.p1_ref.load_state_dict(self.p1.state_dict())
            self.p2_ref.load_state_dict(self.p2.state_dict())
            print(f"✓ reference updated at outer {outer}")

            # ----- inline animation every `visualize_every` outer loops -----
            if (outer % visualize_every) == 0:
                print("🎞  Policy roll-out visualization")
                display(animate_episode(self.p1, self.p2))


class CosineWithWarmup:
    """
    torch.optim.lr_scheduler-like wrapper (no state_dict needed)
    lr(t) = lr_init * f(t)
        warm-up for W steps (linear 0→1),
        then cosine decay to lr_min over T steps.
    """
    def __init__(self, optimizer, warmup_steps, total_steps,
                 lr_min=0.0):
        self.opt = optimizer
        self.W   = warmup_steps
        self.T   = total_steps
        self.lr0 = [g['lr'] for g in optimizer.param_groups]
        self.lr_min = lr_min
        self.step_num = 0

    def step(self):
        self.step_num += 1
        if self.step_num <= self.W:
            scale = self.step_num / float(self.W)
        else:
            prog  = (self.step_num - self.W) / float(max(1, self.T - self.W))
            scale = 0.5 * (1 + math.cos(math.pi * prog))
        for lr0, g in zip(self.lr0, self.opt.param_groups):
            g['lr'] = self.lr_min + (lr0 - self.lr_min) * scale

In [51]:
# Cell 4 : visualize.py
import numpy as np
from IPython.display import HTML

def animate_episode(p1: P1Policy,
                    p2: P2Policy,
                    device: torch.device = device,
                    save_gif: bool = False,
                    fps: int = 5):
    env = HexnersGameEnv(device=device)
    obs = env.reset()
    i_star = env.i_star.item()            # chosen target index (0 or 1)

    p1_xy, p2_xy = [], []

    with torch.no_grad():
        for _ in range(K):
            a1, _, m1 = p1(obs, env.i_star)
            a2, _, m2 = p2(obs)
            step = env.step(a1, a2, m1["j"])
            env.p = bayes_update(env.p, m1["A"], m1["j"])
            obs = step["state"]

            p1_xy.append(env.x[0, 0:2].cpu().numpy())
            p2_xy.append(env.x[0, 4:6].cpu().numpy())

    p1_xy = np.array(p1_xy);   p2_xy = np.array(p2_xy)

    # ---------- Matplotlib -------------------------------------
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(-BOX_POS - .2, BOX_POS + .2)
    ax.set_ylim(-BOX_POS - .2, BOX_POS + .2)
    ax.set_aspect('equal')
    ax.set_title("Hexner's game – one episode")

    # ---- targets: chosen = black star, un-chosen = grey hollow star ----
    for idx, tgt in enumerate(Z_TARGETS.cpu().numpy()):
        if idx == i_star:
            ax.plot(tgt[0], tgt[1], marker='*', ms=14, color='black',
                    label='chosen target' if idx == 0 else None)
        else:
            ax.plot(tgt[0], tgt[1], marker='*', ms=14, color='grey',
                    mfc='none', label='unchosen target' if idx == 1 else None)

    # ---- player markers ----
    p1_scatter = ax.scatter([], [], s=80, color='red',  label='P1')
    p2_scatter = ax.scatter([], [], s=80, color='blue', label='P2')
    ax.legend(loc="upper right")

    # ---- animation helpers ----
    def init():
        p1_scatter.set_offsets(np.empty((0, 2)))
        p2_scatter.set_offsets(np.empty((0, 2)))
        return p1_scatter, p2_scatter

    def update(frame):
        p1_scatter.set_offsets(p1_xy[frame])
        p2_scatter.set_offsets(p2_xy[frame])
        return p1_scatter, p2_scatter

    ani = animation.FuncAnimation(fig, update, frames=K,
                                  init_func=init, blit=True,
                                  interval=1000 / fps)
    plt.close(fig)

    if save_gif:
        ani.save("hexner_episode.gif", writer='pillow', fps=fps)
    return HTML(ani.to_jshtml())

In [52]:
# Cell 5 : main
env_factory = lambda: HexnersGameEnv(device=device)

trainer = RNADTrainer(
    env_factory,
    p1_live=P1Policy(),
    p2_live=P2Policy(),
    value_net=ValueNet()
)

trainer.train(outer_iter=OUTER_ITER,
              inner_epochs=INNER_EPOCH,
              batch_size=2048,    
              visualize_every=1)

[outer 0 | ep 0] Lπ=-0.719  LV=0.068  Adv=+0.0000
[outer 0 | ep 50] Lπ=-0.743  LV=0.062  Adv=-0.0000
[outer 0 | ep 100] Lπ=-0.677  LV=0.060  Adv=-0.0000
[outer 0 | ep 150] Lπ=-0.747  LV=0.063  Adv=-0.0000
[outer 0 | ep 200] Lπ=-0.690  LV=0.059  Adv=+0.0000
[outer 0 | ep 250] Lπ=-0.768  LV=0.060  Adv=+0.0000
[outer 0 | ep 300] Lπ=-0.720  LV=0.060  Adv=-0.0000
[outer 0 | ep 350] Lπ=-0.778  LV=0.060  Adv=+0.0000
[outer 0 | ep 400] Lπ=-0.794  LV=0.059  Adv=-0.0000
[outer 0 | ep 450] Lπ=-0.677  LV=0.059  Adv=-0.0000
[outer 0 | ep 500] Lπ=-0.709  LV=0.060  Adv=+0.0000
[outer 0 | ep 550] Lπ=-0.785  LV=0.065  Adv=-0.0000
[outer 0 | ep 600] Lπ=-0.754  LV=0.060  Adv=+0.0000
[outer 0 | ep 650] Lπ=-0.824  LV=0.062  Adv=+0.0000
[outer 0 | ep 700] Lπ=-0.746  LV=0.064  Adv=+0.0000
[outer 0 | ep 750] Lπ=-0.826  LV=0.064  Adv=-0.0000
[outer 0 | ep 800] Lπ=-0.898  LV=0.064  Adv=+0.0000
[outer 0 | ep 850] Lπ=-0.859  LV=0.064  Adv=+0.0000
[outer 0 | ep 900] Lπ=-0.939  LV=0.067  Adv=+0.0000
[outer 0 | ep 9

[outer 1 | ep 0] Lπ=-0.413  LV=0.032  Adv=+0.0000
[outer 1 | ep 50] Lπ=-0.366  LV=0.033  Adv=+0.0000
[outer 1 | ep 100] Lπ=-0.386  LV=0.032  Adv=-0.0000
[outer 1 | ep 150] Lπ=-0.341  LV=0.032  Adv=-0.0000
[outer 1 | ep 200] Lπ=-0.453  LV=0.030  Adv=+0.0000
[outer 1 | ep 250] Lπ=-0.386  LV=0.033  Adv=-0.0000
[outer 1 | ep 300] Lπ=-0.246  LV=0.032  Adv=+0.0000
[outer 1 | ep 350] Lπ=-0.399  LV=0.030  Adv=-0.0000
[outer 1 | ep 400] Lπ=-0.341  LV=0.030  Adv=-0.0000
[outer 1 | ep 450] Lπ=-0.409  LV=0.031  Adv=+0.0000
[outer 1 | ep 500] Lπ=-0.350  LV=0.031  Adv=-0.0000
[outer 1 | ep 550] Lπ=-0.580  LV=0.034  Adv=-0.0000
[outer 1 | ep 600] Lπ=-0.433  LV=0.032  Adv=+0.0000
[outer 1 | ep 650] Lπ=-0.376  LV=0.032  Adv=+0.0000
[outer 1 | ep 700] Lπ=-0.310  LV=0.031  Adv=+0.0000
[outer 1 | ep 750] Lπ=-0.381  LV=0.032  Adv=+0.0000
[outer 1 | ep 800] Lπ=-0.436  LV=0.035  Adv=-0.0000
[outer 1 | ep 850] Lπ=-0.310  LV=0.032  Adv=+0.0000
[outer 1 | ep 900] Lπ=-0.303  LV=0.034  Adv=-0.0000
[outer 1 | ep 9

[outer 2 | ep 0] Lπ=-0.308  LV=0.033  Adv=-0.0000
[outer 2 | ep 50] Lπ=-0.245  LV=0.033  Adv=-0.0000
[outer 2 | ep 100] Lπ=-0.363  LV=0.033  Adv=-0.0000
[outer 2 | ep 150] Lπ=-0.426  LV=0.033  Adv=+0.0000
[outer 2 | ep 200] Lπ=-0.313  LV=0.035  Adv=+0.0000
[outer 2 | ep 250] Lπ=-0.364  LV=0.036  Adv=-0.0000
[outer 2 | ep 300] Lπ=-0.367  LV=0.036  Adv=-0.0000
[outer 2 | ep 350] Lπ=-0.393  LV=0.037  Adv=+0.0000
[outer 2 | ep 400] Lπ=-0.403  LV=0.036  Adv=-0.0000
[outer 2 | ep 450] Lπ=-0.338  LV=0.033  Adv=+0.0000
[outer 2 | ep 500] Lπ=-0.484  LV=0.035  Adv=-0.0000
[outer 2 | ep 550] Lπ=-0.330  LV=0.035  Adv=-0.0000
[outer 2 | ep 600] Lπ=-0.426  LV=0.033  Adv=-0.0000
[outer 2 | ep 650] Lπ=-0.484  LV=0.034  Adv=+0.0000
[outer 2 | ep 700] Lπ=-0.313  LV=0.033  Adv=+0.0000
[outer 2 | ep 750] Lπ=-0.295  LV=0.034  Adv=+0.0000
[outer 2 | ep 800] Lπ=-0.397  LV=0.036  Adv=+0.0000
[outer 2 | ep 850] Lπ=-0.361  LV=0.035  Adv=+0.0000
[outer 2 | ep 900] Lπ=-0.401  LV=0.035  Adv=-0.0000
[outer 2 | ep 9

[outer 3 | ep 0] Lπ=-0.278  LV=0.035  Adv=+0.0000
[outer 3 | ep 50] Lπ=-0.396  LV=0.035  Adv=+0.0000
[outer 3 | ep 100] Lπ=-0.421  LV=0.036  Adv=-0.0000
[outer 3 | ep 150] Lπ=-0.313  LV=0.037  Adv=+0.0000
[outer 3 | ep 200] Lπ=-0.290  LV=0.032  Adv=-0.0000
[outer 3 | ep 250] Lπ=-0.421  LV=0.036  Adv=+0.0000
[outer 3 | ep 300] Lπ=-0.413  LV=0.035  Adv=+0.0000
[outer 3 | ep 350] Lπ=-0.330  LV=0.034  Adv=+0.0000
[outer 3 | ep 400] Lπ=-0.364  LV=0.037  Adv=+0.0000
[outer 3 | ep 450] Lπ=-0.312  LV=0.034  Adv=-0.0000
[outer 3 | ep 500] Lπ=-0.385  LV=0.032  Adv=-0.0000
[outer 3 | ep 550] Lπ=-0.299  LV=0.034  Adv=+0.0000
[outer 3 | ep 600] Lπ=-0.316  LV=0.034  Adv=+0.0000
[outer 3 | ep 650] Lπ=-0.373  LV=0.034  Adv=-0.0000
[outer 3 | ep 700] Lπ=-0.412  LV=0.033  Adv=+0.0000
[outer 3 | ep 750] Lπ=-0.357  LV=0.036  Adv=-0.0000
[outer 3 | ep 800] Lπ=-0.462  LV=0.035  Adv=-0.0000
[outer 3 | ep 850] Lπ=-0.392  LV=0.035  Adv=+0.0000
[outer 3 | ep 900] Lπ=-0.306  LV=0.036  Adv=-0.0000
[outer 3 | ep 9

KeyboardInterrupt: 

## Code for REINFORCE

In [44]:
# ########################## REINFORCE ##########################

# # Cell 3 : trainer.py
# # ---------------------------------------------------------------
# #  Roll-outs, Bayes update, actor-critic (R-NaD flavour)
# # ---------------------------------------------------------------


# # ---------------------------------------------------------------
# class HexnerTrainer:
#     def __init__(self,
#                  env: HexnersGameEnv,
#                  p1: P1Policy,
#                  p2: P2Policy,
#                  critic: ValueNet,
#                  lr: float = 3e-4,
#                  entropy_beta: Tuple[float,float] = (1e-2, 1e-2)):
#         self.env    = env
#         self.p1     = p1.to(device)
#         self.p2     = p2.to(device)
#         self.critic = critic.to(device)
        
#         # 2 separate optims (can be merged if you prefer)
#         self.opt_p  = torch.optim.Adam(
#             list(self.p1.parameters()) +
#             list(self.p2.parameters()), lr=lr)
#         self.opt_v  = torch.optim.Adam(self.critic.parameters(), lr=lr)
        
#         self.β1, self.β2 = entropy_beta
    
#     # ------------------------------------------------------------
#     def rollout_batch(self, batch_size: int):
#         env = self.env
#         env.batch_size = batch_size
#         obs = env.reset()
        
#         # Storage
#         logp1, logp2        = [], []
#         ent1,  ent2         = [], []
#         values              = []
#         running_costs       = []
        
#         i_star = env.i_star                          # (B,)
        
#         for k in range(K):
#             # ---- Forward policies ---------------------------------
#             u1, lp1, misc1 = self.p1(obs, i_star)
#             u2, lp2, misc2 = self.p2(obs)
            
#             # Evaluate entropy (for regularisation only)
#             A_rows   = misc1["A"].gather(
#                           1, i_star.view(batch_size,1,1).expand(-1,1,I)
#                        ).squeeze(1)                  # (B,I)
#             ent1_t   = -(A_rows * torch.log(A_rows+1e-8)).sum(-1)  # (B,)
#             ent2_t   = -(misc2["B"] * torch.log(misc2["B"]+1e-8)).sum(-1)
            
#             # ---- Environment step --------------------------------
#             step_out = env.step(u1, u2, misc1["j"])
#             running_costs.append(step_out["running_cost"])   # (B,)
            
#             # Bayes update of public belief p
#             env.p = bayes_update(env.p, misc1["A"], misc1["j"])
#             obs = step_out["state"]                          # new obs
            
#             # Logging
#             logp1.append(lp1)
#             logp2.append(lp2)
#             ent1.append(ent1_t)
#             ent2.append(ent2_t)
#             values.append(self.critic(obs))         # (B,)
        
#         # Terminal cost
#         g = env._terminal_cost()                 # (B,)
        
#         # Stack tensors time-first
#         logp1 = torch.stack(logp1, dim=0)        # (K,B)
#         logp2 = torch.stack(logp2, dim=0)
#         ent1  = torch.stack(ent1 , dim=0)
#         ent2  = torch.stack(ent2 , dim=0)
#         rcost = torch.stack(running_costs, dim=0)
#         values= torch.stack(values,  dim=0)      # V(s_{t+1})
        
#         return {
#             "logp1": logp1, "logp2": logp2,
#             "entropy1": ent1, "entropy2": ent2,
#             "running": rcost,
#             "values": values,
#             "terminal": g,
#             "i_star": i_star,
#         }
    
#     # ------------------------------------------------------------
#     def train_step(self, batch_size: int = 256, clip_grad: float = 5.0):
#         roll = self.rollout_batch(batch_size)
        
#         # ---- Compute returns ------------------------------------
#         #  G = Σ_t l_t + g
#         G = roll["running"].sum(dim=0) + roll["terminal"]     # (B,)
        
#         # Advantage: A = G − V̂(s₀)   (use critic at final state baseline)
#         baseline = roll["values"][0]                          # (B,)
#         Adv = (G - baseline).detach()
        
#         # Critic loss (MSE)
#         v_loss = F.mse_loss(baseline, G)
        
#         # ---- Policy losses -------------------------------------
#         # Player-1 minimises ⇒ uses +Adv
#         J1 = (roll["logp1"].sum(0) * Adv).mean() - \
#              self.β1 * roll["entropy1"].sum(0).mean()
#         # Player-2 maximises ⇒ uses −Adv
#         J2 = -(roll["logp2"].sum(0) * Adv).mean() - \
#              self.β2 * roll["entropy2"].sum(0).mean()
#         policy_loss = J1 + J2
        
#         # ---- Optimise ------------------------------------------
#         self.opt_p.zero_grad()
#         policy_loss.backward()
#         torch.nn.utils.clip_grad_norm_(self.p1.parameters(), clip_grad)
#         torch.nn.utils.clip_grad_norm_(self.p2.parameters(), clip_grad)
#         self.opt_p.step()
        
#         self.opt_v.zero_grad()
#         v_loss.backward()
#         torch.nn.utils.clip_grad_norm_(self.critic.parameters(), clip_grad)
#         self.opt_v.step()
        
#         return {"policy_loss": policy_loss.item(),
#                 "v_loss": v_loss.item(),
#                 "adv_mean": Adv.mean().item()}